# Connect to PostgreSQL

In [15]:
import psycopg2

try:
   # Connect to PostgreSQL
   connection = psycopg2.connect(
       dbname="Evolution",
       user="ev",
       password="Temp@123",
       host="192.168.4.51", # or your server's IP address
       port="5432" # default PostgreSQL port
   )
   print("Connection established successfully!")
except Exception as e:
   print(f"Error: {e}")


Connection established successfully!


In [16]:
cur = connection.cursor()

In [25]:
s = """SELECT deals.id, deals.name, deals_units.buildingid, deals_customers.id AS customer_id 
			FROM deals JOIN deals_customers ON deals.id = deals_customers.deal_id JOIN deals_units ON deals.id = deals_units.deal_id 
            WHERE deals_customers.id = %s AND deals_units.buildingid = %s"""

params = [1798, '9', 0, 100]

In [28]:
rows = cur.execute(s, (1892, '9'))

In [29]:
cur.fetchall()

[(1859, 'D-002040-Hyde-1704', '9', 1892)]

# Load the data

In [3]:
import pandas as pd
import os
import json


In [4]:
directory = "C:/Users/maryam.maksour/OneDrive - AL BAYARI/Desktop/RAG/DATA"


In [21]:
table_columns_path = 'table_columns.json'
with open(table_columns_path, 'r', encoding='utf-8-sig') as file:
                table_columns = json.load(file)

In [5]:
text_data_path = 'text_data.json'
with open(text_data_path, 'r', encoding='utf-8-sig') as file:
                text_data = json.load(file)

In [22]:
def pre_row(row, table_name):
    for col in table_columns[table_name]:
        if col not in row or row[col] == '-' or row[col] == '' or row[col] == ' ':
            
            if table_columns[table_name][col] == "int" or table_columns[table_name][col] == "float":
                    row[col] = ' 0 '
            else:
                 row[col] = ' Null '
    return row

In [23]:
def row2text():

	return eval(text_data)
        

# Load the embedding model

In [5]:
from langchain_ollama import OllamaEmbeddings

emb = OllamaEmbeddings(model="bge-large", base_url="http://192.168.43.220:11435")

def generate_embedding(text): 
    text = str(text)
    embedding = emb.embed_query(text)
    return embedding

c:\Users\maryam.maksour\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
full_path = os.path.join(directory, "deals.json")
file_name = "Deals"
if os.path.isfile(full_path):
    with open(full_path, 'r', encoding='utf-8-sig') as file:
        data = json.load(file)

        i = 0
      
        for row in data[file_name]: 
                if row['CreationDate'] == None:
                     row['CreationDate'] = '2024-01-01T00:00:00Z'
                row['CreationDate'] = row['CreationDate'].replace(' ', '')
                
                x = cur.execute("""
						UPDATE deals
						SET CreationDate = %s
						WHERE id = %s;
						""", (row['CreationDate'], row['Id'])) 
                i += 1
                if i % 100 == 0:
                                        print(f"Updated {i} rows.")
                                        connection.commit()

Updated 100 rows.
Updated 200 rows.
Updated 300 rows.
Updated 400 rows.
Updated 500 rows.
Updated 600 rows.
Updated 700 rows.
Updated 800 rows.
Updated 900 rows.
Updated 1000 rows.
Updated 1100 rows.
Updated 1200 rows.
Updated 1300 rows.
Updated 1400 rows.
Updated 1500 rows.
Updated 1600 rows.
Updated 1700 rows.
Updated 1800 rows.
Updated 1900 rows.
Updated 2000 rows.


In [20]:

connection.commit()
connection.rollback()

In [ ]:
def _ensure_No_embed_in_select(sql: str) :

    s = str(sql).lower()
    s = (s or "").strip()
    s = s.split()

    for i in range (len(s)):
        if s[i] == "select":
            while s[i] != 'from':
                i += 1
                if s[i].find("embed") != -1:
                    return ValueError("can not select embed column")
                
    return sql

In [37]:
s = "SELECT buildings.id, buildings.name, buildings.shortname, buildings.address, buildings.location, projects.name AS project_name, developers.name AS developer_name, buildings.status FROM buildings LEFT JOIN projects ON buildings.projectid = projects.id LEFT JOIN developers ON buildings.developerid = developers.id WHERE ((buildings.embed_name <=> $1::vector) < 0.35 OR (buildings.embed_shortname <=> $1::vector) < 0.35) AND buildings.status ILIKE $2 ORDER BY buildings.name ASC LIMIT $3 OFFSET $4"

In [50]:
_ensure_No_embed_in_select(s)

['select', 'buildings.id,', 'buildings.name,', 'buildings.shortname,', 'buildings.address,', 'buildings.location,', 'projects.name', 'as', 'project_name,', 'developers.name', 'as', 'developer_name,', 'buildings.status', 'from', 'buildings', 'left', 'join', 'projects', 'on', 'buildings.projectid', '=', 'projects.id', 'left', 'join', 'developers', 'on', 'buildings.developerid', '=', 'developers.id', 'where', '((buildings.embed_name', '<=>', '$1::vector)', '<', '0.35', 'or', '(buildings.embed_shortname', '<=>', '$1::vector)', '<', '0.35)', 'and', 'buildings.status', 'ilike', '$2', 'order', 'by', 'buildings.name', 'asc', 'limit', '$3', 'offset', '$4']


'SELECT buildings.id, buildings.name, buildings.shortname, buildings.address, buildings.location, projects.name AS project_name, developers.name AS developer_name, buildings.status FROM buildings LEFT JOIN projects ON buildings.projectid = projects.id LEFT JOIN developers ON buildings.developerid = developers.id WHERE ((buildings.embed_name <=> $1::vector) < 0.35 OR (buildings.embed_shortname <=> $1::vector) < 0.35) AND buildings.status ILIKE $2 ORDER BY buildings.name ASC LIMIT $3 OFFSET $4'